# Gold Layer - Product Performance Fact Table

## Purpose
Analyze product-level performance metrics for business analytics and product management.

## Type
**Fact Table** (materialized, product metrics)

## Input
* **Source:** `big_data.silver.order_products` (33.8M rows)
* **Source:** `big_data.silver.products_enriched` (49.7K rows)

## Output
* **Target:** `big_data.gold.ft_product_performance`
* **Rows:** 49K (one per product)
* **Primary Key:** product_id
* **Columns:** product_id, product_name, department, aisle, price_usd, times_ordered, times_reordered, estimated_revenue_usd, reorder_rate

## Transformations

### Step 1: Aggregate Product Metrics
* JOIN order_products with products_enriched
* GROUP BY product_id to calculate:
  * times_ordered (COUNT)
  * times_reordered (SUM of reordered flag)
  * estimated_revenue_usd (SUM of price_usd)
  * reorder_rate (% of items reordered)

## Data Quality Validations

### Technical Validations
* Row count > 40,000 products
* NOT NULL on product_id, times_ordered

### Business Validations
* Revenue > 0 for all products

## Why Materialized?
* ✅ 49K rows (large fact table)
* ✅ Used frequently in dashboards
* ✅ Complex aggregations
* ✅ Product analysis queries

## Persistence
Only persists if **all validations pass**.

## Execution
Expected runtime: ~3-5 minutes.

In [0]:
%run ../UTILS/data_quality_checks

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
silver_schema = "big_data.silver"
gold_schema = "big_data.gold"

# Target table (fact table with ft_ prefix)
target_table = "ft_product_performance"

# Expected counts for validation
expected_metrics = {
    "min_products": 40_000
}

# Create Gold schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print("Configuration:")
print(f"  Source schema: {silver_schema}")
print(f"  Target: {gold_schema}.{target_table}")

In [0]:
print("Step 1: Aggregating...")

order_products = spark.table(f"{silver_schema}.order_products")
products = spark.table(f"{silver_schema}.products_enriched")

product_perf = order_products.join(products, "product_id", "left").groupBy("product_id", "product_name", "department", "aisle", "price_usd").agg(
    F.count("*").alias("times_ordered"),
    F.sum(F.when(F.col("reordered") == True, 1).otherwise(0)).alias("times_reordered"),
    F.round(F.sum("price_usd"), 2).alias("estimated_revenue_usd"),
    F.round(F.sum(F.when(F.col("reordered") == True, 1).otherwise(0)) / F.count("*") * 100, 2).alias("reorder_rate")
).withColumn("_gold_timestamp", F.current_timestamp())

print(f"  Products: {product_perf.count():,}")

In [0]:
print("Step 2: Aggregating department metrics...")

# Aggregate by department
dept_perf_gold = product_perf.groupBy("department").agg(
    F.sum("times_ordered").alias("total_orders"),
    F.round(F.sum("estimated_revenue_usd"), 2).alias("total_revenue_usd"),
    F.round(F.avg("reorder_rate"), 2).alias("avg_reorder_rate")
).withColumn("_gold_timestamp", F.current_timestamp())

print(f"  Departments: {dept_perf_gold.count():,}")
print("\nPreview - Department Performance (Top 5 by Revenue):")
dept_perf_gold.orderBy(F.desc("total_revenue_usd")).show(5, truncate=False)

In [0]:
print_validation_header("product_performance - Technical Validations")

# Initialize validation flag
validation_passed_technical = True

# 1. Product count check
product_count = product_perf.count()
print(f"\nTotal products: {product_count:,}")
print(f"Expected: >= {expected_metrics['min_products']:,}\n")

if product_count >= expected_metrics["min_products"]:
    status = "PASS"
    msg = f"Product count ({product_count:,}) >= {expected_metrics['min_products']:,}"
else:
    status = "FAIL"
    msg = f"Product count ({product_count:,}) < {expected_metrics['min_products']:,}"
    validation_passed_technical = False
print_check_result("PRODUCT COUNT", status, msg)

# 2. NOT NULL checks
status, failed, msg = check_not_null(product_perf, ["product_id", "times_ordered"])
print_check_result("NOT NULL (product_id, times_ordered)", status, msg, failed)
if status == "FAIL":
    validation_passed_technical = False

# 3. Department count check
dept_count = dept_perf_gold.count()
print(f"\nDepartment count: {dept_count}")
print(f"Expected: {expected_metrics['expected_departments']}\n")

if dept_count == expected_metrics["expected_departments"]:
    status = "PASS"
    msg = f"Department count matches expected ({dept_count})"
else:
    status = "FAIL"
    msg = f"Department count mismatch: {dept_count} vs {expected_metrics['expected_departments']}"
    validation_passed_technical = False
print_check_result("DEPARTMENT COUNT", status, msg)

print("\n" + "="*60)
if validation_passed_technical:
    print("SUCCESS: Technical validations PASSED")
else:
    print("FAILURE: Technical validations FAILED")
print("="*60)

In [0]:
print_validation_header("product_performance - Business Validations")

# Initialize business validation flag
validation_passed_business = True

# 1. All products have positive revenue
print("\n1. Business Rule - Revenue Validation:")
negative_revenue_count = product_perf.filter(F.col("estimated_revenue_usd") <= 0).count()

if negative_revenue_count == 0:
    status = "PASS"
    msg = "All products have positive revenue"
else:
    status = "FAIL"
    msg = f"{negative_revenue_count} products with non-positive revenue"
    validation_passed_business = False
print_check_result("REVENUE > 0", status, msg, negative_revenue_count)

# 2. Department coverage
print("\n2. Business Rule - Department Coverage:")
top_depts = dept_perf_gold.orderBy(F.desc("total_revenue_usd")).limit(5).collect()
print("\n  Top 5 Departments by Revenue:")
for dept in top_depts:
    print(f"    - {dept['department']}: ${dept['total_revenue_usd']:,.2f}")

status = "PASS"
msg = f"All {expected_metrics['expected_departments']} departments have data"
print_check_result("DEPARTMENT COVERAGE", status, msg)

print("\n" + "="*60)
if validation_passed_business:
    print("SUCCESS: Business validations PASSED")
else:
    print("FAILURE: Business validations FAILED")
print("="*60)

In [0]:
validation_passed = validation_passed_technical and validation_passed_business
print("\n" + "="*60)
print("OVERALL VALIDATION")
print(f"  Technical: {'PASSED' if validation_passed_technical else 'FAILED'}")
print(f"  Business:  {'PASSED' if validation_passed_business else 'FAILED'}")
print("="*60)

In [0]:
# Only persist if validation passed
if validation_passed:
    print("Persisting to Gold layer...\n")
    
    # Save ft_product_performance
    product_perf.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{gold_schema}.{target_table}")
    
    # Verify
    product_count = spark.table(f"{gold_schema}.{target_table}").count()
    
    print("\n" + "="*60)
    print("SUCCESS: Product performance table persisted to Gold layer")
    print("="*60)
    print(f"\nFinal Statistics:")
    print(f"  Table: {gold_schema}.{target_table}")
    print(f"  Rows: {product_count:,}")
    print(f"  Type: Fact table (product metrics)")
    print(f"  Format: Delta")
    print(f"\nNext Step: Query for product analytics and insights")
else:
    print("\n" + "="*60)
    print("ABORTED: Validation failed - table NOT persisted")
    print("="*60)
    print("\nFix validation errors above and re-run.")